In [2]:
import numpy as np
import pandas as pd


In [6]:
data=pd.read_csv('Dataset/customer_shopping_behavior.csv')

In [7]:
data.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [5]:
data.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

In [8]:
data['Review Rating']=data.groupby('Category')['Review Rating'].transform(lambda x:x.fillna(x.median()))

In [9]:
data.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

In [11]:
# Converting hidder in lower case 
data.columns=data.columns.str.lower()

In [12]:
# Replacing space with UnderScore
data.columns=data.columns.str.replace(" ","_")

In [18]:
# data.columns=data.columns.str.replace('purchase_amount_(usd)','purchase_amount')
data=data.rename(columns={'purchase_amount_(usd)':'purchase_amount'})

In [17]:
data.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

In [19]:
#create a column age_group
labels=['Young Adult','Adult','Middle Aged','Senior']
data['age_group']=pd.qcut(data['age'],q=4,labels=labels)

In [21]:
data[['age','age_group']].head(10)

,age,age_group
0,55,Middle Aged
1,19,Young Adult
2,50,Middle Aged
3,21,Young Adult
4,45,Middle Aged
5,46,Middle Aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle Aged


In [23]:
# create column purchase frequency days
frequency_mapping={
'Fortnightly':14,
'Weekly'     :7,
'Monthly'    : 30,
'Quarterly'  : 90,
'Bi-Weekly'  : 14,
'Annually'   :365,
'Every 3 Months' :90

}
data['purchases_frequency_days']=data['frequency_of_purchases'].map(frequency_mapping)

In [27]:
data[['purchases_frequency_days','frequency_of_purchases']].head(10)

,purchases_frequency_days,frequency_of_purchases
0,14,Fortnightly
1,14,Fortnightly
2,7,Weekly
3,7,Weekly
4,365,Annually
5,7,Weekly
6,90,Quarterly
7,7,Weekly
8,365,Annually
9,90,Quarterly


In [29]:
data[['discount_applied', 'promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [30]:
(data['discount_applied']==data['promo_code_used']).all()

np.True_

In [31]:
data=data.drop('promo_code_used',axis=1)

In [32]:
data.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchases_frequency_days'],
      dtype='object')

In [33]:
pip install psycopg2-binary sqlalchemy

Note: you may need to restart the kernel to use updated packages.


In [34]:
from sqlalchemy import create_engine

# Step 1: Connect to PostgreSQL
# Replace placeholders with your actual details

username="postgres"   # default user
password="1234"       # the password you set during installtion
host    ="localhost"  #  if running locally
port    ="5432"       # default postgres port
database="customer_behavior" # the database you created in pgamin

engine=create_engine(f"postgresql+psycopg2://{username}:{password}@{host}/{database}")


# Step 2: Load Dataframe into postgresSQL

table_name='customer'   # choose any table name
data.to_sql(table_name,engine,if_exists='replace',index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Data successfully loaded into table 'customer' in database 'customer_behavior'.
